## Advmod count

The aim is to create an example table that shows ADVMOD *deprel* frequency in transactions by verb and absolute frequency of verbs in transactions. In addition, the table shows relative and absolute differences between verb and ADVMOD frequencies.

This information can be used in determining whether a specific *deprel* value is a required member of a verb rection. It can be further extended to grammatical cases etc.

In [10]:
import sys
sys.path.append('../../../common_code')

In [11]:
import sqlite3
from db_operations.db_display import *

## Input parameters

In [12]:
INPUT_DIR = "C:/Users/liivas/Documents/Töö/verbirektisoonid"

RESULT_DB = "advmod_count.db"
TRANSACTION_DB = f"{INPUT_DIR}/v32_data.db"

## Data processing

In [3]:
con = sqlite3.connect(TRANSACTION_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{RESULT_DB}" AS adv')

cur.execute("""
DROP TABLE IF EXISTS adv.advmod_count
""")

cur.execute("""
CREATE TABLE adv.advmod_count AS
SELECT
    tbl1.verb,
    tbl1.verb_compound,
    advmod_count,
    count(*) AS verb_count,
    (CAST(count(*) AS REAL) - CAST(advmod_count AS REAL)) / count(*) * 100 AS relative_diff,
    count(*) - advmod_count AS absolute_diff
FROM
(
    SELECT
        verb,
        verb_compound
    FROM
        transaction_head as tr_head
) as tbl1
INNER JOIN
(
    SELECT
        verb,
        verb_compound,
        count(*) AS advmod_count
    FROM
    (
        SELECT
            id,
            verb,
            verb_compound
        FROM
            transaction_head as tr_head2
    ) AS tbl_verb
    INNER JOIN
    (
        SELECT DISTINCT
            head_id
        FROM
            transaction_row AS tr_row
        WHERE
            tr_row.deprel='advmod'
    ) AS tbl_advmod
    ON
        tbl_verb.id=tbl_advmod.head_id
    GROUP BY
        verb, verb_compound
) AS tbl2
ON
    tbl1.verb = tbl2.verb AND tbl1.verb_compound = tbl2.verb_compound
GROUP BY
    tbl1.verb, tbl1.verb_compound
ORDER BY
    relative_diff ASC
""")

con.close()

## Result

In [13]:
display_sqlite_as_dataframe(RESULT_DB, 'advmod_count', 10)

,verb,verb_compound,advmod_count,verb_count,relative_diff,absolute_diff
0,002kutsuma,,2,2,0.0,0
1,09rõõmustama,,1,1,0.0,0
2,10kordama,,1,1,0.0,0
3,1991pooldama,,1,1,0.0,0
4,1arvama,,1,1,0.0,0
5,1pagema,,1,1,0.0,0
6,2009hakkama,,1,1,0.0,0
7,20kordistuma,,1,1,0.0,0
8,20saama,,1,1,0.0,0
9,30GB-segama,,2,2,0.0,0
